In [1]:
import os, sys

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['PYSPARK_PYTHON'] = sys.executable

print('Python:', sys.version)
print('JAVA_HOME:', os.environ.get('JAVA_HOME'))


Python: 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataCleaningBikes") \
    .master("local[*]") \
    .getOrCreate()

print("Spark started successfully")
print("Spark version:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/28 21:38:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark started successfully
Spark version: 3.5.0


## Libraries

In [3]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when, isnan, count, lit
from pyspark.sql.functions import *
from pyspark.sql.types import ( StringType, DoubleType, IntegerType, FloatType, LongType)
from pyspark.ml.feature import StringIndexer


In [4]:
# Load Data
df = spark.read.csv(
    "/home/vboxuser/Big Data Systems Design/Afeena Final Project/Data Cleaning/Bikes_raw.csv",
    header=True,
    inferSchema=True
)

df.show(1)
df.printSchema()

26/03/28 21:38:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+---------------+---------------+--------------------+--------+---------+--------+-------+-------+--------+-------------------+-----------+------------+----------+----------+----------+-----------+--------+--------------------+-------------+---------+----------+---------+----------+-----------+---------+------+--------+--------------------+--------+--------------------+------------+-----------+-----------+-----------+
|OBJECTID|EVENT_UNIQUE_ID|PRIMARY_OFFENCE|            OCC_DATE|OCC_YEAR|OCC_MONTH| OCC_DOW|OCC_DAY|OCC_DOY|OCC_HOUR|        REPORT_DATE|REPORT_YEAR|REPORT_MONTH|REPORT_DOW|REPORT_DAY|REPORT_DOY|REPORT_HOUR|DIVISION|       LOCATION_TYPE|PREMISES_TYPE|BIKE_MAKE|BIKE_MODEL|BIKE_TYPE|BIKE_SPEED|BIKE_COLOUR|BIKE_COST|STATUS|HOOD_158|   NEIGHBOURHOOD_158|HOOD_140|   NEIGHBOURHOOD_140|  LONG_WGS84|  LAT_WGS84|          x|          y|
+--------+---------------+---------------+--------------------+--------+---------+--------+-------+-------+--------+-------------------+--

In [5]:
# Check for nulls
null_counts = df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])

#print full results with column names
null_counts.show(truncate=False)


[Stage 3:=============================>                             (2 + 2) / 4]

+--------+---------------+---------------+--------+--------+---------+-------+-------+-------+--------+-----------+-----------+------------+----------+----------+----------+-----------+--------+-------------+-------------+---------+----------+---------+----------+-----------+---------+------+--------+-----------------+--------+-----------------+----------+---------+---+---+
|OBJECTID|EVENT_UNIQUE_ID|PRIMARY_OFFENCE|OCC_DATE|OCC_YEAR|OCC_MONTH|OCC_DOW|OCC_DAY|OCC_DOY|OCC_HOUR|REPORT_DATE|REPORT_YEAR|REPORT_MONTH|REPORT_DOW|REPORT_DAY|REPORT_DOY|REPORT_HOUR|DIVISION|LOCATION_TYPE|PREMISES_TYPE|BIKE_MAKE|BIKE_MODEL|BIKE_TYPE|BIKE_SPEED|BIKE_COLOUR|BIKE_COST|STATUS|HOOD_158|NEIGHBOURHOOD_158|HOOD_140|NEIGHBOURHOOD_140|LONG_WGS84|LAT_WGS84|x  |y  |
+--------+---------------+---------------+--------+--------+---------+-------+-------+-------+--------+-----------+-----------+------------+----------+----------+----------+-----------+--------+-------------+-------------+---------+----------+---

### Handle outliers: BIKE_COST

In [6]:
# IQR detection
q1, q3 = df.approxQuantile("BIKE_COST", [0.25, 0.75], 0.01)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

bike_outliers_before = df.filter(
    (F.col("BIKE_COST") < lower) | (F.col("BIKE_COST") > upper)
).count()
print(f"BIKE_COST outliers BEFORE: {bike_outliers_before}")


# Cap at 1st/99th percentile instead of deleting
caps = df.approxQuantile("BIKE_COST", [0.01, 0.99], 0.01)
lower_floor, upper_cap = caps

df = df.withColumn(
    "BIKE_COST",
    F.when(F.col("BIKE_COST") > upper_cap, upper_cap)
    .when(F.col("BIKE_COST") < lower_floor, lower_floor)
    .otherwise(F.col("BIKE_COST"))
)

lower_floor, upper_cap = df.approxQuantile("BIKE_COST", [0.01, 0.99], 0.01)
outliers_after = df.filter(
    (F.col("BIKE_COST") < lower_floor) | (F.col("BIKE_COST") > upper_cap)
).count()
print(f"BIKE_COST outliers AFTER (percentile cap): {outliers_after}")

# Recheck outliers
lower_floor, upper_cap = df.approxQuantile("BIKE_COST", [0.01, 0.99], 0.01)

outliers_after = df.filter(
    (F.col("BIKE_COST") < lower_floor) | (F.col("BIKE_COST") > upper_cap)
).count()

print(f"BIKE_COST outliers AFTER (percentile method): {outliers_after}")


BIKE_COST outliers BEFORE: 3316
BIKE_COST outliers AFTER (percentile cap): 0
BIKE_COST outliers AFTER (percentile method): 0


In [7]:
# ************BEFORE COUNTS

before_counts = {}

# Geographic bounds
before_counts["LONG_WGS84"] = df.filter(
    (F.col("LONG_WGS84") < -79.7) | (F.col("LONG_WGS84") > -79.1)
).count()

before_counts["LAT_WGS84"] = df.filter(
    (F.col("LAT_WGS84") < 43.5) | (F.col("LAT_WGS84") > 43.9)
).count()

# X, Y (invalid if 0 or extreme via IQR)
x_q1, x_q3 = df.approxQuantile("x", [0.25, 0.75], 0.01)
x_iqr = x_q3 - x_q1
x_lower = x_q1 - 1.5 * x_iqr
x_upper = x_q3 + 1.5 * x_iqr

before_counts["x"] = df.filter(
    (F.col("x") == 0) | (F.col("x") < x_lower) | (F.col("x") > x_upper)
).count()

y_q1, y_q3 = df.approxQuantile("y", [0.25, 0.75], 0.01)
y_iqr = y_q3 - y_q1
y_lower = y_q1 - 1.5 * y_iqr
y_upper = y_q3 + 1.5 * y_iqr

before_counts["y"] = df.filter(
    (F.col("y") == 0) | (F.col("y") < y_lower) | (F.col("y") > y_upper)
).count()

# REPORT_HOUR
before_counts["REPORT_HOUR"] = df.filter(
    (F.col("REPORT_HOUR") < 0) | (F.col("REPORT_HOUR") > 23)
).count()

# OCC_YEAR
before_counts["OCC_YEAR"] = df.filter(
    (F.col("OCC_YEAR") < 1975) | (F.col("OCC_YEAR") > 2025)
).count()


print("BEFORE CLEANING:")
for k, v in before_counts.items():
    print(f"{k}: {v}")


# ************CLEAN

# Geographic
df = df.withColumn(
    "LONG_WGS84",
    F.when((F.col("LONG_WGS84") < -79.7) | (F.col("LONG_WGS84") > -79.1), None)
    .otherwise(F.col("LONG_WGS84"))
)

df = df.withColumn(
    "LAT_WGS84",
    F.when((F.col("LAT_WGS84") < 43.5) | (F.col("LAT_WGS84") > 43.9), None)
    .otherwise(F.col("LAT_WGS84"))
)

# X, Y
df = df.withColumn(
    "x",
    F.when((F.col("x") == 0) | (F.col("x") < x_lower) | (F.col("x") > x_upper), None)
    .otherwise(F.col("x"))
)

df = df.withColumn(
    "y",
    F.when((F.col("y") == 0) | (F.col("y") < y_lower) | (F.col("y") > y_upper), None)
    .otherwise(F.col("y"))
)

# REPORT_HOUR
median_hour = df.approxQuantile("REPORT_HOUR", [0.5], 0.01)[0]

df = df.withColumn(
    "REPORT_HOUR",
    F.when((F.col("REPORT_HOUR") < 0) | (F.col("REPORT_HOUR") > 23), median_hour)
    .otherwise(F.col("REPORT_HOUR"))
)

# OCC_YEAR
df = df.withColumn(
    "OCC_YEAR",
    F.when((F.col("OCC_YEAR") < 1975) | (F.col("OCC_YEAR") > 2025), None)
    .otherwise(F.col("OCC_YEAR"))
)

# ************AFTER COUNTS

after_counts = {}

after_counts["LONG_WGS84"] = df.filter(
    (F.col("LONG_WGS84") < -79.7) | (F.col("LONG_WGS84") > -79.1)
).count()

after_counts["LAT_WGS84"] = df.filter(
    (F.col("LAT_WGS84") < 43.5) | (F.col("LAT_WGS84") > 43.9)
).count()

after_counts["x"] = df.filter(
    (F.col("x") == 0) | (F.col("x") < x_lower) | (F.col("x") > x_upper)
).count()

after_counts["y"] = df.filter(
    (F.col("y") == 0) | (F.col("y") < y_lower) | (F.col("y") > y_upper)
).count()

after_counts["REPORT_HOUR"] = df.filter(
    (F.col("REPORT_HOUR") < 0) | (F.col("REPORT_HOUR") > 23)
).count()

after_counts["OCC_YEAR"] = df.filter(
    (F.col("OCC_YEAR") < 1975) | (F.col("OCC_YEAR") > 2025)
).count()


print("\nAFTER CLEANING:")
for k, v in after_counts.items():
    print(f"{k}: {v}")



print("\nCOMPARISON (Before → After):")
for col in before_counts:
    print(f"{col}: {before_counts[col]} → {after_counts[col]}")

BEFORE CLEANING:
LONG_WGS84: 347
LAT_WGS84: 348
x: 5851
y: 5131
REPORT_HOUR: 0
OCC_YEAR: 0

AFTER CLEANING:
LONG_WGS84: 0
LAT_WGS84: 0
x: 0
y: 0
REPORT_HOUR: 0
OCC_YEAR: 0

COMPARISON (Before → After):
LONG_WGS84: 347 → 0
LAT_WGS84: 348 → 0
x: 5851 → 0
y: 5131 → 0
REPORT_HOUR: 0 → 0
OCC_YEAR: 0 → 0


## IMPUTATION

In [8]:
from pyspark.sql import functions as F

# Fix numeric columns first
df = df.withColumn("BIKE_COST", F.expr("try_cast(BIKE_COST as double)")) \
       .withColumn("BIKE_SPEED", F.expr("try_cast(BIKE_SPEED as double)"))

# Calculate MEDIAN for BIKE_COST, MEAN for BIKE_SPEED
median_cost = df.approxQuantile("BIKE_COST", [0.5], 0.01)[0]
mean_speed = df.select(F.avg("BIKE_SPEED")).collect()[0][0]

print(f"Median BIKE_COST: ${median_cost:.0f}")
print(f"Mean BIKE_SPEED: {mean_speed:.1f}")

# Fill missing BIKE_COST with median
df = df.withColumn(
    "BIKE_COST",
    F.when(F.col("BIKE_COST").isNull(), median_cost)
    .otherwise(F.col("BIKE_COST"))
)

# Fill missing BIKE_SPEED with mean
df = df.withColumn(
    "BIKE_SPEED",
    F.when(F.col("BIKE_SPEED").isNull(), mean_speed)
    .otherwise(F.col("BIKE_SPEED"))
)

# Rest stays the same
GARBAGE = ["", "0", "-", "?", "(UNK)"]

def is_missing_str(c):
    cond = F.col(c).isNull()
    for g in GARBAGE:
        cond = cond | (F.col(c) == g)
    return cond

df = df.withColumn(
    "BIKE_MAKE",
    F.when(is_missing_str("BIKE_MAKE"), "No make").otherwise(F.col("BIKE_MAKE"))
)

df = df.withColumn(
    "BIKE_MODEL",
    F.when(is_missing_str("BIKE_MODEL"), "No model").otherwise(F.col("BIKE_MODEL"))
)

df = df.withColumn(
    "BIKE_COLOUR",
    F.when(is_missing_str("BIKE_COLOUR"), "No colour").otherwise(F.col("BIKE_COLOUR"))
)

df.select("BIKE_COST", "BIKE_SPEED", "BIKE_MAKE", "BIKE_MODEL", "BIKE_COLOUR").show(10, truncate=False)

Median BIKE_COST: $699
Mean BIKE_SPEED: 14.1
+---------+----------+----------+-----------+-----------+
|BIKE_COST|BIKE_SPEED|BIKE_MAKE |BIKE_MODEL |BIKE_COLOUR|
+---------+----------+----------+-----------+-----------+
|1300.0   |21.0      |FELT      |F59        |SILRED     |
|699.0    |10.0      |SUPERCYCLE|No model   |No colour  |
|699.0    |1.0       |TREK      |SOHO S     |BLK        |
|1019.0   |9.0       |GI        |TCX2 (2010)|BLU        |
|400.0    |25.0      |BI        |No model   |RED        |
|1500.0   |21.0      |CA        |RZ 120 1   |WHI        |
|750.0    |21.0      |NORCO     |CHARGER 9.2|BLK        |
|500.0    |24.0      |KHS       |VITAMIN A  |WHI        |
|1500.0   |18.0      |KO        |No model   |BLK        |
|1200.0   |18.0      |OT        |No model   |BLK        |
+---------+----------+----------+-----------+-----------+
only showing top 10 rows



### Verify: no missing values remain in key columns

In [9]:
df.select(
    F.count(F.when(F.col("BIKE_COST").isNull() | F.isnan("BIKE_COST"), 1)).alias("BIKE_COST_missing"),
    F.count(F.when(F.col("BIKE_SPEED").isNull() | F.isnan("BIKE_SPEED"), 1)).alias("BIKE_SPEED_missing"),
    F.count(F.when(is_missing_str("BIKE_MAKE"), 1)).alias("BIKE_MAKE_missing"),
    F.count(F.when(is_missing_str("BIKE_MODEL"), 1)).alias("BIKE_MODEL_missing"),
    F.count(F.when(is_missing_str("BIKE_COLOUR"), 1)).alias("BIKE_COLOUR_missing"),
).show()


+-----------------+------------------+-----------------+------------------+-------------------+
|BIKE_COST_missing|BIKE_SPEED_missing|BIKE_MAKE_missing|BIKE_MODEL_missing|BIKE_COLOUR_missing|
+-----------------+------------------+-----------------+------------------+-------------------+
|                0|                 0|                0|                 0|                  0|
+-----------------+------------------+-----------------+------------------+-------------------+



## Feature Engineer- Occurence Period (Moning, Afternoon, Evening, Night)

In [10]:
df = df.withColumn(
    "OCCURENCE_PERIOD",
    F.when((F.col("OCC_HOUR") >= 6)  & (F.col("OCC_HOUR") <= 12), "Morning")
     .when((F.col("OCC_HOUR") > 12)  & (F.col("OCC_HOUR") <= 17), "Afternoon")
     .when((F.col("OCC_HOUR") > 17)  & (F.col("OCC_HOUR") <= 20), "Evening")
     .otherwise("Night")
)
df.groupBy("OCCURENCE_PERIOD").count().orderBy("count", ascending=False).show()


+----------------+-----+
|OCCURENCE_PERIOD|count|
+----------------+-----+
|           Night|11119|
|         Morning|10946|
|       Afternoon|10539|
|         Evening| 7244|
+----------------+-----+



## Feature Engineer- Risk_Level

In [11]:
theft_counts = df.filter(F.col("STATUS") == "STOLEN") \
    .groupBy("NEIGHBOURHOOD_158") \
    .agg(F.count("*").alias("Total_Thefts"))

# Data-driven thresholds using 33rd and 66th percentiles
low_thresh, high_thresh = theft_counts.approxQuantile("Total_Thefts", [0.33, 0.66], 0.01)
print(f"Risk thresholds — Low: <{low_thresh:.0f}, Medium: {low_thresh:.0f}-{high_thresh:.0f}, High: >{high_thresh:.0f}")

theft_counts = theft_counts.withColumn(
    "RISK_LEVEL",
    F.when(F.col("Total_Thefts") > high_thresh, "High")
     .when(F.col("Total_Thefts") >= low_thresh,  "Medium")
     .otherwise("Low")
)

df = df.join(
    theft_counts.select("NEIGHBOURHOOD_158", "RISK_LEVEL"),
    on="NEIGHBOURHOOD_158",
    how="left"
)

# Fill nulls from the left join- rows whose neighbourhood had no STOLEN
# records set to 'Unknown' instead of silently remaining null.
df = df.withColumn(
    "RISK_LEVEL",
    F.when(F.col("Risk_Level").isNull(), "Unknown")
     .otherwise(F.col("Risk_Level"))
)

df.groupBy("RISK_LEVEL").count().orderBy("RISK_LEVEL").show()


Risk thresholds — Low: <65, Medium: 65-170, High: >170


+----------+-----+
|RISK_LEVEL|count|
+----------+-----+
|      High|32076|
|       Low| 1811|
|    Medium| 5960|
|   Unknown|    1|
+----------+-----+



## Feature Engineer- High Cardinality Reduction (Make/Model)

In [12]:
# Clean Bike Make and Models.

# Clean models
top_models = df.groupBy("BIKE_MODEL").count().sort(F.desc("count")).limit(100)
top_models_list = [row['BIKE_MODEL'] for row in top_models.collect()]

#print(top_models)
#print(top_models_list)

# Replace weird models with "OTHER"
df = df.withColumn(
    "BIKE_MODEL_CLEAN",
    F.when(F.col("BIKE_MODEL").isin(top_models_list), F.col("BIKE_MODEL")).otherwise("OTHER")
)


# Clean Make
top_makes = df.groupBy("BIKE_MAKE").count().sort(F.desc("count")).limit(100)
top_makes_list = [row['BIKE_MAKE'] for row in top_makes.collect()]

# print(top_makes)
# print(top_makes_list)

# Replace weird models with "OTHER"
df = df.withColumn(
    "BIKE_MAKE_CLEAN",
    F.when(F.col("BIKE_MAKE").isin(top_makes_list), F.col("BIKE_MAKE")).otherwise("OTHER")
)

df= df.drop("BIKE_MODEL","BIKE_MAKE")
df = df.withColumnsRenamed({
    "BIKE_MODEL_CLEAN" : "BIKE_MODEL", 
    "BIKE_MAKE_CLEAN": "BIKE_MAKE"}
)
df.printSchema()

root
 |-- NEIGHBOURHOOD_158: string (nullable = true)
 |-- OBJECTID: integer (nullable = true)
 |-- EVENT_UNIQUE_ID: string (nullable = true)
 |-- PRIMARY_OFFENCE: string (nullable = true)
 |-- OCC_DATE: string (nullable = true)
 |-- OCC_YEAR: integer (nullable = true)
 |-- OCC_MONTH: string (nullable = true)
 |-- OCC_DOW: string (nullable = true)
 |-- OCC_DAY: integer (nullable = true)
 |-- OCC_DOY: integer (nullable = true)
 |-- OCC_HOUR: integer (nullable = true)
 |-- REPORT_DATE: string (nullable = true)
 |-- REPORT_YEAR: integer (nullable = true)
 |-- REPORT_MONTH: string (nullable = true)
 |-- REPORT_DOW: string (nullable = true)
 |-- REPORT_DAY: integer (nullable = true)
 |-- REPORT_DOY: integer (nullable = true)
 |-- REPORT_HOUR: double (nullable = true)
 |-- DIVISION: string (nullable = true)
 |-- LOCATION_TYPE: string (nullable = true)
 |-- PREMISES_TYPE: string (nullable = true)
 |-- BIKE_TYPE: string (nullable = true)
 |-- BIKE_SPEED: double (nullable = true)
 |-- BIKE_COLO

In [13]:
# Remove duplicates
print("Before duplicates:", df.count())
df = df.dropDuplicates()
print("After duplicates: ", df.count())


Before duplicates: 39848


[Stage 90:=============================>                            (2 + 2) / 4]

After duplicates:  39848


### Drop identifier and raw date columns

In [14]:
# Remove strings that have no value to model
df = df.drop("OBJECTID", "EVENT_UNIQUE_ID", "OCC_DATE", "REPORT_DATE")


### Final null check after all cleaning

In [15]:
null_counts_final = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
null_counts_final.show(truncate=False)


[Stage 99:=============================>                            (2 + 2) / 4]

+-----------------+---------------+--------+---------+-------+-------+-------+--------+-----------+------------+----------+----------+----------+-----------+--------+-------------+-------------+---------+----------+-----------+---------+------+--------+--------+-----------------+----------+---------+----+----+----------------+----------+----------+---------+
|NEIGHBOURHOOD_158|PRIMARY_OFFENCE|OCC_YEAR|OCC_MONTH|OCC_DOW|OCC_DAY|OCC_DOY|OCC_HOUR|REPORT_YEAR|REPORT_MONTH|REPORT_DOW|REPORT_DAY|REPORT_DOY|REPORT_HOUR|DIVISION|LOCATION_TYPE|PREMISES_TYPE|BIKE_TYPE|BIKE_SPEED|BIKE_COLOUR|BIKE_COST|STATUS|HOOD_158|HOOD_140|NEIGHBOURHOOD_140|LONG_WGS84|LAT_WGS84|x   |y   |OCCURENCE_PERIOD|RISK_LEVEL|BIKE_MODEL|BIKE_MAKE|
+-----------------+---------------+--------+---------+-------+-------+-------+--------+-----------+------------+----------+----------+----------+-----------+--------+-------------+-------------+---------+----------+-----------+---------+------+--------+--------+----------------

### Drop rows with missing GPS coordinates

In [16]:
count_before = df.count()
print(f"Row count before GPS drop: {count_before}")

df_cleaned = df.dropna(subset=["LONG_WGS84", "LAT_WGS84", "x", "y"])

count_after = df_cleaned.count()
print(f"Row count after GPS drop:  {count_after}")
print(f"Rows removed:              {count_before - count_after}")


Row count before GPS drop: 39848


[Stage 117:==========================================>              (3 + 1) / 4]

Row count after GPS drop:  31558
Rows removed:              8290


In [17]:
# Check nulls again

null_counts_final = df_cleaned.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
null_counts_final.show(truncate=False)

+-----------------+---------------+--------+---------+-------+-------+-------+--------+-----------+------------+----------+----------+----------+-----------+--------+-------------+-------------+---------+----------+-----------+---------+------+--------+--------+-----------------+----------+---------+---+---+----------------+----------+----------+---------+
|NEIGHBOURHOOD_158|PRIMARY_OFFENCE|OCC_YEAR|OCC_MONTH|OCC_DOW|OCC_DAY|OCC_DOY|OCC_HOUR|REPORT_YEAR|REPORT_MONTH|REPORT_DOW|REPORT_DAY|REPORT_DOY|REPORT_HOUR|DIVISION|LOCATION_TYPE|PREMISES_TYPE|BIKE_TYPE|BIKE_SPEED|BIKE_COLOUR|BIKE_COST|STATUS|HOOD_158|HOOD_140|NEIGHBOURHOOD_140|LONG_WGS84|LAT_WGS84|x  |y  |OCCURENCE_PERIOD|RISK_LEVEL|BIKE_MODEL|BIKE_MAKE|
+-----------------+---------------+--------+---------+-------+-------+-------+--------+-----------+------------+----------+----------+----------+-----------+--------+-------------+-------------+---------+----------+-----------+---------+------+--------+--------+-----------------+--

### Save cleaned data to CSV

In [18]:
output_directory = "/home/vboxuser/Big Data Systems Design/Afeena Final Project/Cleaned Bikes"
file_name = "cleaned_bike_data.csv"
full_path = os.path.join(output_directory, file_name)

if not os.path.exists(output_directory):
    os.makedirs(output_directory)
    print(f"Created directory: {output_directory}")

df_cleaned.coalesce(1).write.format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .save(full_path)

print(f"Saved to: {os.path.abspath(full_path)}")


[Stage 137:>                                                        (0 + 1) / 1]

Saved to: /home/vboxuser/Big Data Systems Design/Afeena Final Project/Cleaned Bikes/cleaned_bike_data.csv


In [19]:
spark.stop()

In [20]:
# df_cleaned.show(1, truncate=False)
df_cleaned.printSchema()


root
 |-- NEIGHBOURHOOD_158: string (nullable = true)
 |-- PRIMARY_OFFENCE: string (nullable = true)
 |-- OCC_YEAR: integer (nullable = true)
 |-- OCC_MONTH: string (nullable = true)
 |-- OCC_DOW: string (nullable = true)
 |-- OCC_DAY: integer (nullable = true)
 |-- OCC_DOY: integer (nullable = true)
 |-- OCC_HOUR: integer (nullable = true)
 |-- REPORT_YEAR: integer (nullable = true)
 |-- REPORT_MONTH: string (nullable = true)
 |-- REPORT_DOW: string (nullable = true)
 |-- REPORT_DAY: integer (nullable = true)
 |-- REPORT_DOY: integer (nullable = true)
 |-- REPORT_HOUR: double (nullable = true)
 |-- DIVISION: string (nullable = true)
 |-- LOCATION_TYPE: string (nullable = true)
 |-- PREMISES_TYPE: string (nullable = true)
 |-- BIKE_TYPE: string (nullable = true)
 |-- BIKE_SPEED: double (nullable = true)
 |-- BIKE_COLOUR: string (nullable = true)
 |-- BIKE_COST: double (nullable = true)
 |-- STATUS: string (nullable = true)
 |-- HOOD_158: string (nullable = true)
 |-- HOOD_140: string (